# CSET Assessment

In [77]:
import os 
import json 
import regex as re
from collections import Counter, defaultdict
from datetime import datetime

from utils import read_word_list, get_category_names

In [3]:
papers = []
with open(os.path.join('..', 'data', 'arxiv_data.jsonl'), 'r') as file:
    for line in file: 
        papers.append(json.loads(line))

In [78]:
cs_category_map = get_category_names('https://arxiv.org/archive/cs')

/Users/haleyjohnson/Desktop/haley_johnson_cset_assessment/code/utils.py:33: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 33 of the file /Users/haleyjohnson/Desktop/haley_johnson_cset_assessment/code/utils.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  '''


## Question 1: Identifying papers about agentic AI

### Question 1.A

In [ ]:
# load in pre-defined list of words
narrow_word_list = read_word_list(os.path.join('..', 'word_lists', 'agentic_ai_narrow.txt'))
broad_word_list = read_word_list(os.path.join('..', 'word_lists', 'agentic_ai_broad.txt'))

In [ ]:
narrow_matches = set()
broad_matches = set()

# compiling regular expressions outside of loop to improve runtime & remove redundant operations
narrow_regex_patterns = [re.compile(f'(?:\\b)?{word}(:?\\b)?') for word in narrow_word_list]
broad_regex_patterns = [re.compile(f'(?:\\b)?{word}(:?\\b)?') for word in broad_word_list]

for paper in papers:
    title = paper.get('title', '').lower()
    abstract = paper.get('abstract', '').lower()
    paper_id = paper.get('id')

    for pattern in narrow_regex_patterns:
        if pattern.search(title) or pattern.search(abstract):
            narrow_matches.add(paper_id)
            # only need to match one of the patterns to count 
            # cut down run time by ending loop if a match is already found
            break

    for pattern in broad_regex_patterns:
        if pattern.search(title) or pattern.search(abstract):
            broad_matches.add(paper_id)
            # only need to match one of the patterns to count 
            # cut down run time by ending loop if a match is already found
            break

### Question 1.B

In [40]:
cs_ma_papers = set([paper.get('id') for paper in papers if 'cs.ma' in paper.get('categories').lower()])

In [43]:
only_in_cs_ma = cs_ma_papers - broad_matches
only_in_broad_matches = broad_matches - cs_ma_papers
items_not_shared = broad_matches ^ cs_ma_papers

In [ ]:
# quick check that set math was done correctly
len(items_not_shared) == (len(only_in_broad_matches) + len(only_in_cs_ma))

True

What repositories were relevant papers in? 

In [81]:
repository_counts = Counter()

for paper in papers: 
    if paper.get('id') in broad_matches:
        repos = paper.get('categories_split')

        for repo in repos:
            repository_counts.update([cs_category_map.get(repo, repo)])

In [82]:
repository_counts.most_common(10)

[('Artificial Intelligence', 4079),
 ('Multiagent Systems', 1861),
 ('Computation and Language', 1690),
 ('Machine Learning', 1656),
 ('Systems and Control', 893),
 ('eess.SY', 817),
 ('Robotics', 727),
 ('Cryptography and Security', 493),
 ('Software Engineering', 481),
 ('math.OC', 478)]

## Question 2: Growth in research on agentic AI

In [ ]:
publications_by_year = defaultdict(set)
cs_publications_by_year = defaultdict(set)
agentic_ai_publications_by_year = defaultdict(set)

for paper in papers:
    paper_id = paper.get('id')
    date = paper.get('created')
    year = datetime.fromisoformat(date).year

    publications_by_year[year].add(paper_id)

    if paper_id in broad_matches:
        agentic_ai_publications_by_year[year].add(paper_id)

    if 'cs' in paper.get('categories'):
        cs_publications_by_year[year].add(paper_id)


In [75]:
pct_agentic_ai_publications = {}

for year in list(publications_by_year.keys()):
    agentic_ai_pubs = len(agentic_ai_publications_by_year.get(year, set()))
    overall_pubs = len(publications_by_year.get(year, set()))

    pct_agentic_ai_publications[year] = (agentic_ai_pubs / overall_pubs)

In [76]:
pct_agentic_ai_publications

{2023: 0.0024173055780054315,
 2022: 0.0018843172331544425,
 2024: 0.004866148475922297,
 2025: 0.009767088048963191,
 2026: 0.015824809917893208,
 2021: 0.0019090118245964495,
 2019: 0.0014477100903095342,
 2020: 0.0016357526682752802,
 2017: 0.0008612234157077582,
 2011: 0.0005117489018721481,
 2018: 0.0014124027796086703,
 2016: 0.0007969879352516009,
 2010: 0.00023228803716608595,
 2015: 0.0008794274623968948,
 2005: 0.0002106741573033708,
 2014: 0.0008589078656139539,
 2009: 0.00015264068383026355,
 2012: 0.0005481168271865946,
 2013: 0.0004545772431638576,
 2008: 0.0005533115697449233,
 1999: 0.00011872254541137362,
 2007: 0.00011747430249632892,
 2003: 0.0004229760595550292,
 1996: 0.0,
 1997: 0.0,
 2006: 0.0004626263961403741,
 2004: 0.00030014256771966684,
 2000: 0.00032209576980888986,
 1998: 0.0,
 1995: 0.0,
 1993: 0.0,
 2001: 0.0,
 2002: 0.00018335166850018335,
 1994: 0.0,
 1992: 0.0,
 1991: 0.0,
 1989: 0.0,
 1990: 0.0}